# HYCOM ESPC-D-V02 下载

按真实 UTC 时间分月，支持 T/S/U/V、分块续传、指数退避重试、超时终止、自动拆块和数据有效性检查。

**运行顺序：导入 → 配置 → 预览 → 下载 → 查看报告。** 核心实现位于同目录 `hycom_download.py`；原 notebook 保存在 `archive/Download_HYCOM.original.ipynb`。

依赖：`numpy pandas xarray netCDF4 tqdm ipywidgets`，不需要 dask。首次安装可在当前 notebook 内运行 `%pip install -r requirements-download.txt`。

数据源：[HYCOM ESPC-D-V02 官方说明](https://www.hycom.org/dataserver/espc-d-v02/global-analysis)。本下载器使用年度 `t3z/s3z/u3z/v3z` OPeNDAP 聚合，适用于你使用的 2024 年 9 月之后的数据。


In [ ]:
from pathlib import Path
import sys
import logging

# 兼容从 HYCOM 目录或其父目录启动 Jupyter。
module_dir = Path.cwd() if (Path.cwd() / "hycom_download.py").exists() else Path.cwd() / "HYCOM"
if not (module_dir / "hycom_download.py").exists():
    raise FileNotFoundError("请从 HYCOM 目录或其父目录启动 Jupyter；notebook 与 hycom_download.py 保持同目录。")
if str(module_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(module_dir.resolve()))

from hycom_download import DownloadConfig, QCConfig, Downloader
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)


## 1. 集中配置

时间使用 UTC，`start` 包含，`end` 不包含。例如 2024 年 9 月：`start="2024-09-01", end="2024-10-01"`。

默认采用公司慢网络配置：一步一块、数据操作等待 180 秒、元数据等待 300 秒。保留原来下载 2025 年 U/V 的默认选择；可改为 `("T", "S", "U", "V")`。输出根目录下会自动创建各年份、各变量目录。为避免与旧月文件混用，默认写入新的 `HYCOM_download_v2` 目录。

水平及深度范围向外取最近格点以完整覆盖，例如深度上限 3800 m 可能选到 4000 m 层。经度必须采用服务器坐标制；跨日期变更线请分两次请求。


In [ ]:
config = DownloadConfig(
    start="2025-01-01",
    end="2026-01-01",
    variables=("U", "V"),                   # 可选 T / S / U / V
    out_root=module_dir / "HYCOM_download_v2", # 可改成 Path(r"D:\HYCOM_download_v2")
    lon_range=(116.0, 123.0),
    lat_range=(20.5, 27.5),
    depth_range=(0.0, 3800.0),

    chunk_steps=1,                          # 公司慢网络：一步一块；稳定后可调 4/8/16
    max_chunk_mb=32,                        # 自动缩小时间块；临时内存会高于此值
    max_attempts=4,                         # 每个块的总尝试次数
    timeout_seconds=180,                    # 数据块读取超时（秒）
    metadata_timeout_seconds=300,           # 元数据/完整坐标读取单独等待（秒）
    max_consecutive_remote_failures=8,       # 公司网络长时间不可用时暂停本轮数据源
    backoff_seconds=2,
    max_backoff_seconds=60,
    compression_level=1,
    show_progress=True,                    # 时间步进度、耗时、预计剩余时间；同时保留日志

    qc=QCConfig(
        raw_zero_limits={"T": 0.20, "S": 0.20, "U": 0.01, "V": 0.01}, # 原始零值比例上限
        min_valid_fraction=0.0,             # 0=每时刻至少一个有效点，允许陆地/深层空值
        physical_ranges={},                # 例如 {"T": (-3, 40)}，覆盖默认宽松范围
    ),
)
downloader = Downloader(config)


## 2. 先预览

只读取坐标和变量元数据，不下载四维场。检查时间范围、数据量、输出路径和 `missing_steps`。

**缺少任何请求时刻的月份不会被标为完成，也不会自动补 NaN。** 特别是年度聚合可能不从 1 月 1 日 00:00 开始；以预览为准，核对数据源覆盖或调整明确的时间范围。重新配置后重新创建 `Downloader(config)`。


In [ ]:
plan = downloader.preview()
display(plan)
missing = plan.loc[plan["missing_steps"] > 0]
if not missing.empty:
    print("以下月份数据源覆盖不完整，下载时会报告 failed：")
    display(missing[["variable", "month", "missing_steps", "first_time", "last_time"]])


## 3. 开始 / 继续下载

运行下面单元开始下载。中断后重新运行即可续传。失败块耗尽重试后自动拆半，直到单时刻；成功块会保留。

输出文件名兼容原处理流程，例如 `HYCOM_2025_monthly_U_qc/HYCOM_2025_U_M09.nc`。旧流程文件没有请求指纹，不能仅凭名称判定可复用；同名冲突会报告失败并保留旧文件，请使用独立输出目录。

月文件只有全部数据通过检查才原子写入；`.parts` 保存可续传块，`.complete.json` 保存完成凭证与 SHA-256。修改范围、时间或 QC 后会重新验证请求；不支持两个 notebook 同时写同一输出目录。

每个变量、每个月显示已处理时间步、百分比、耗时、预计剩余时间及当前 UTC 数据时刻，区分成功/缓存/失败，单列重试次数。重试不会重复推进进度。**100% 只表示时间步已处理，存在失败时仍不会合并月文件**；以进度条最终状态和报告为准。元数据阶段使用日志。


In [ ]:
report = downloader.run()
display(report)


## 4. 查看未完成月份

`complete` = 本次完成；`cached` = 已验证并跳过；`incomplete` = 个别时刻持续失败；`failed` = 覆盖、配置或文件操作错误。

根目录 `download_report.json` 每个月更新一次；不完整月份的 `.parts/.../failures.json` 列出实际失败时间和原因。修复配置或服务端恢复后重新运行。坏时刻不在下载阶段插值，以保留原始数据的可追溯性。


In [ ]:
unfinished = report.loc[~report["status"].isin(["complete", "cached"])]
if unfinished.empty:
    print("请求范围内的所有月份均已完成并通过检查。")
else:
    display(unfinished[["variable", "month", "status", "detail"]])
    print("重新运行下载单元可续传；改动 config 后要先重建 downloader。")


## 检查范围与注意事项

- 校验变量、维度、完整坐标、时间单位、每时刻是否空场、全部深度物理范围，以及 T/S/U/V 最浅选中层 raw==0 比例（T/S 20%，U/V 1%）。
- 零值比例基于解码前的 raw，排除 `_FillValue/missing_value`；物理范围检查才应用 `scale_factor/add_offset`。下载文件保留原始打包值，避免重复缩放。
- `raw==0` 是异常启发式，不能证明服务端所有数值都正确；温度真实接近 20℃ 或流速接近 0 时需结合物理背景判断。可按变量配置阈值，必要时通过 `raw_zero_limits={}` 关闭该项。
- 永久陆地允许为空，但整块所选海域在某时刻全缺测会失败；纯陆地区域不适合此下载器。
- 月文件按真实日历分组，首尾不足整月时只包含请求时段；后续脚本需采用对应时间范围。
- 网络在单独子进程中运行以实现可终止超时，使用 `.py` 脚本调用时将 `run()` 放进 `if __name__ == "__main__":`。本 notebook 可直接逐单元运行。
- 详细说明见 `README_download.md`；离线测试：在 HYCOM 目录执行 `python -m unittest discover -s tests -v`。
